# 06 — Final Integration
Merge saved outputs into dashboard-ready data (no model inference).

### 1. Restore
Upload the **four final ZIPs** from Stages 02–05.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import json, shutil, pandas as pd

archives = [
    "Maharah_Processed_Data.zip", "Maharah_Diarization_Results.zip",
    "Maharah_Summarization_Results_FINAL.zip", "Maharah_Sentiment_Results_FINAL.zip"
]
files.upload()
for name in archives:
    assert Path(name).is_file(), f"Missing ZIP: {name}"
    with ZipFile(name) as z:
        z.extractall("/content")
print("Restored:", len(archives), "ZIPs")

### 2. Sources
Validate 12 calls and load evaluation tables.

In [ ]:
base = Path("/content/data/processed")
ASR = base / "asr/small_en_cpu"
DIA = base / "diarization/community_1"
SUM = base / "summarization/model_comparison/qwen3_4b_2507"
SEN = base / "sentiment/reference_assisted"
ids = [f"CALL_{i:03d}" for i in range(1, 13)]

required = [base / "selected_calls.csv", DIA / "diarization_metrics.csv",
            SEN / "call_sentiment_results.csv", SEN / "customer_turn_sentiment.csv",
            SEN / "speaker_attribution_review.csv", SEN / "methodology.json"]
required += [p for cid in ids for p in (ASR / f"{cid}.json",
             DIA / f"{cid}_turns.csv", SUM / f"{cid}_summary.json")]
missing = [str(p.relative_to(base)) for p in required if not p.is_file()]
assert not missing, f"Missing files: {missing}"

def indexed(path):
    return (pd.read_csv(path).drop_duplicates("call_id").set_index("call_id")
            if path and path.is_file() else pd.DataFrame())

def score_row(df, cid):
    return json.loads(df.loc[cid].to_json()) if cid in df.index else {}

manifest = indexed(base / "selected_calls.csv")
sentiment = indexed(SEN / "call_sentiment_results.csv")
diar_scores = indexed(DIA / "diarization_metrics.csv")
sum_scores = indexed(base / "summarization/model_comparison/model_quality_comparison_12_calls.csv")
turn_sent = pd.read_csv(SEN / "customer_turn_sentiment.csv")

asr_scores, asr_score_path = pd.DataFrame(), None
for p in sorted(ASR.rglob("*.csv")):
    if "experiments" in p.parts: continue
    try: df = pd.read_csv(p)
    except Exception: continue
    if "call_id" in df and any("wer" in c.lower() for c in df):
        asr_scores = df.drop_duplicates("call_id").set_index("call_id")
        asr_score_path = p
        break

assert set(manifest.index) == set(sentiment.index) == set(ids)
assert set(turn_sent.call_id) == set(ids)
print(f"Calls: {len(ids)} | Customer sentiment turns: {len(turn_sent)}")

### 3. Merge
Keep diarized turns and reference-assisted sentiment turns separate.

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

calls, all_turns = [], []
for cid in ids:
    a, q = read_json(ASR / f"{cid}.json"), read_json(SUM / f"{cid}_summary.json")
    turns = pd.read_csv(DIA / f"{cid}_turns.csv")
    assert {"start", "end", "speaker", "role", "text"} <= set(turns.columns)
    turns.insert(0, "call_id", cid)
    all_turns.append(turns)

    summary = q.get("generated_summary", q.get("summary", {}))
    if isinstance(summary, str):
        try: summary = json.loads(summary)
        except json.JSONDecodeError: summary = {"summary": summary}
    if not isinstance(summary, dict): summary = {"summary": str(summary)}

    text = a.get("predicted_transcript") or a.get("transcript") or " ".join(
        x.get("text", "") for x in a.get("predicted_segments", []))
    calls.append({
        "call_id": cid, "metadata": score_row(manifest, cid),
        "speakers": sorted(set(turns.role.dropna().astype(str))),
        "transcript": text,
        "speaker_turns": json.loads(turns.to_json(orient="records")),
        "summary": summary,
        "sentiment": score_row(sentiment, cid),
        "customer_sentiment_turns": json.loads(turn_sent[turn_sent.call_id.eq(cid)].to_json(orient="records")),
        "evaluation": {"asr": score_row(asr_scores, cid),
                       "diarization": score_row(diar_scores, cid),
                       "summarization": score_row(sum_scores, cid)},
        "limitations": [x for x in (summary.get("notes_limitations"),
                         "Speaker role mapping and sentiment alignment are reference-assisted.") if x]
    })

assert all(c["transcript"] and c["summary"].get("summary") and
           c["sentiment"].get("overall_sentiment") and c["speaker_turns"] for c in calls)
print("Integrated:", len(calls), "complete calls")

### 4. Save
Export JSON, CSV, QA data, and provenance for Streamlit.

In [ ]:
OUT = base / "integrated"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "calls.json").write_text(json.dumps(calls, ensure_ascii=False, indent=2), encoding="utf-8")

flat = []
for c in calls:
    s, t, m, e = c["summary"], c["sentiment"], c["metadata"], c["evaluation"]
    flat.append({"call_id": c["call_id"], "speakers": ", ".join(c["speakers"]),
                 "domain": m.get("domain"), "accent": m.get("accent"),
                 "duration_sec": m.get("duration_sec"), "transcript": c["transcript"],
                 **{k: s.get(k) for k in ("customer_issue", "agent_action", "final_outcome",
                                           "summary", "notes_limitations")},
                 "notes_limitations": "; ".join(c["limitations"]),
                 "overall_sentiment": t.get("overall_sentiment"),
                 "justification": t.get("justification") or t.get("overall_evidence"),
                 "opening_sentiment": t.get("opening_sentiment"),
                 "closing_sentiment": t.get("closing_sentiment"),
                 "sentiment_change": t.get("sentiment_change"),
                 "asr_wer": next((v for k,v in e["asr"].items()
                                  if k.lower() in ("wer", "original_wer")), None),
                 "diarization_der": next((v for k,v in e["diarization"].items()
                                          if k.lower() in ("der", "der_pct", "der_percent")), None),
                 "qwen3_quality": e["summarization"].get("q3_overall")})

pd.DataFrame(flat).to_csv(OUT / "calls.csv", index=False)
pd.concat(all_turns, ignore_index=True).to_csv(OUT / "speaker_turns.csv", index=False)
turn_sent.to_csv(OUT / "customer_turn_sentiment.csv", index=False)
for name in ("speaker_attribution_review.csv", "methodology.json"):
    shutil.copy2(SEN / name, OUT / name)
for p in filter(None, [asr_score_path, DIA / "diarization_metrics.csv",
                       base / "summarization/model_comparison/model_quality_comparison_12_calls.csv",
                       base / "summarization/model_comparison/final_quality_metrics.csv"]):
    if p.is_file(): shutil.copy2(p, OUT / p.name)

provenance = {"calls": len(calls), "customer_sentiment_turns": len(turn_sent),
              "sources": {"asr": "Whisper Small.en", "diarization": "Pyannote Community-1",
                          "summary": "Qwen3-4B-Instruct-2507",
                          "sentiment": read_json(SEN / "methodology.json").get("sentiment_model", "RoBERTa")},
              "limitations": "Speaker roles and sentiment alignment are reference-assisted; models are not rerun."}
(OUT / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
print("Saved:", sorted(p.name for p in OUT.iterdir()))

### 5. Check
Review completeness and sentiment distribution.

In [ ]:
df = pd.read_csv(OUT / "calls.csv")
assert len(df) == 12 and df.call_id.is_unique
assert df[["transcript", "summary", "overall_sentiment", "qwen3_quality"]].notna().all().all()
assert all({"agent", "customer"} <= set(c["speakers"]) for c in calls)
print("Sentiment:", df.overall_sentiment.value_counts().to_dict())
print("Turns:", len(pd.read_csv(OUT / "speaker_turns.csv")), "diarized /",
      len(turn_sent), "customer sentiment")
display(df[["call_id", "domain", "overall_sentiment", "qwen3_quality"]])

### 6. Download
Save the ZIP for GitHub and the Streamlit app.

In [ ]:
zip_name = "/content/Maharah_Integration_Results_FINAL.zip"
with ZipFile(zip_name, "w", ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file(): z.write(p, p.relative_to("/content"))
print("Created:", zip_name)
files.download(zip_name)